# Session 7 · dplyr verbs in pandas

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

The data is the **synthetic** 2025 season from Session 3 (`data/matches_2025_synthetic.csv`): invented results for practice only. Solutions are in the matching `_solutions` notebook.

Today is about **writing pandas the way you write dplyr**: one readable pipeline that reads top to bottom, rather than a dozen lines that each overwrite `df`.

In [38]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [39]:
import numpy as np
import pandas as pd

matches = pd.read_csv("../data/matches_2025_synthetic.csv", parse_dates=["date"])
matches = matches.assign(
    home_score=lambda d: d["home_goals"] * 6 + d["home_behinds"],
    away_score=lambda d: d["away_goals"] * 6 + d["away_behinds"],
    total_goals=lambda d: d["home_goals"] + d["away_goals"],
    margin=lambda d: d["home_score"] - d["away_score"],
)
matches.head(3)

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd,home_score,away_score,total_goals,margin
0,1,2025,1,2025-03-16,Collingwood,Carlton,12,12,19,11,14893.0,84,125,31,-41
1,2,2025,1,2025-03-16,Hawthorn,Brisbane,14,9,17,10,12927.0,93,112,31,-19
2,3,2025,1,2025-03-15,North Melbourne,Richmond,16,11,10,9,21940.0,107,69,26,38


## 1. `.loc` and `.iloc`: R's `df[rows, cols]`

| R | pandas | Selects by |
| --- | --- | --- |
| `df[df$round == 1, c("home_team", "away_team")]` | `df.loc[df["round"] == 1, ["home_team", "away_team"]]` | labels and True/False masks |
| `df[3, "home_team"]` | `df.iloc[2]["home_team"]` or `df.iloc[2, 4]` | **positions**, starting at 0 |

`.loc` is the one you will use 95% of the time. `:` means "all", so `df.loc[:, ["a", "b"]]` selects columns from every row.

In [40]:
matches.loc[matches["round"] == 1, ["home_team", "away_team", "home_score"]]

,home_team,away_team,home_score
0,Collingwood,Carlton,84
1,Hawthorn,Brisbane,93
2,North Melbourne,Richmond,107
3,Essendon,West Coast,72
4,GWS,Fremantle,34
5,Sydney,Western Bulldogs,71
6,Melbourne,St Kilda,72
7,Port Adelaide,Gold Coast,103
8,Adelaide,Geelong,89


In [41]:
matches.iloc[0:3, 0:5]   # first three rows, first five columns (positions, end excluded)

,match_id,season,round,date,home_team
0,1,2025,1,2025-03-16,Collingwood
1,2,2025,1,2025-03-16,Hawthorn
2,3,2025,1,2025-03-15,North Melbourne


### Your turn 1.1

- `geelong_cols`: Geelong's home matches, with only the `round` and `away_team` columns (use `.loc`)
- `third_row_team`: the home team in the **third** row (use `.iloc`; remember positions start at 0)

In [42]:
geelong_cols = matches.loc[matches["home_team"] == "Geelong", ["round", "away_team"]]
third_row_team = matches.iloc[2]["home_team"]

In [43]:
check("1.1 geelong_cols shape", lambda: geelong_cols.shape, (11, 2))
check("1.1 third_row_team", lambda: third_row_team, 'North Melbourne')

PASS     1.1 geelong_cols shape
PASS     1.1 third_row_team


## 2. Method chaining: pandas' pipe

Wrap the whole pipeline in brackets, start each step on a new line with a dot, and it reads like a dplyr chain:

| dplyr | pandas chain step |
| --- | --- |
| `mutate(x = a + b)` | `.assign(x=lambda d: d["a"] + d["b"])` |
| `filter(round <= 5)` | `.query("round <= 5")` or `.loc[lambda d: d["round"] <= 5]` |
| `select(a, b)` | `.loc[:, ["a", "b"]]` or `.filter(["a", "b"])` |
| `arrange(desc(a))` | `.sort_values("a", ascending=False)` |
| `slice_head(n = 5)` | `.head(5)` |

The `lambda d:` inside `assign` means "the data as it is at this step", like the `.data` pronoun in dplyr. Without it, you would refer to the original `matches` and miss any columns added earlier in the chain.

In [44]:
big_home_wins = (
    matches
    .assign(win_by_10_goals=lambda d: d["margin"] >= 60)
    .query("win_by_10_goals")
    .loc[:, ["round", "home_team", "away_team", "margin"]]
    .sort_values("margin", ascending=False)
)
big_home_wins.head()

,round,home_team,away_team,margin
58,7,Melbourne,Western Bulldogs,104
13,2,Melbourne,Richmond,96
151,17,North Melbourne,Collingwood,82
198,23,GWS,Essendon,79
200,23,Geelong,Port Adelaide,78


### Your turn 2.1

Write **one chain** that builds `top_away`:

1. Keep rounds 10 to 15 inclusive (`query` understands `10 <= round <= 15`)
2. Keep only `round`, `away_team` and `away_score`
3. Sort by `away_score`, highest first
4. Keep the top 5 rows

In [45]:
top_away = (
    matches
    # add your four steps here, each starting with a dot
    .query("10 <= round <= 15")
    .loc[:, ["round", "away_team", "away_score"]]
    .sort_values("away_score", ascending=False)
    .iloc[0:5, :]
)

In [46]:
check("2.1 top_away scores", lambda: list(top_away["away_score"]), [133, 130, 115, 112, 111])

PASS     2.1 top_away scores


## 3. `case_when`, lookups and renaming

| dplyr | pandas |
| --- | --- |
| `ifelse(cond, a, b)` | `np.where(cond, a, b)` |
| `case_when(c1 ~ a, c2 ~ b, TRUE ~ z)` | `np.select([c1, c2], [a, b], default=z)` |
| a named-vector lookup, `lookup[x]` | `s.map(lookup_dict)` |
| `rename(new = old)` | `.rename(columns={"old": "new"})` (note: old first) |

`np.select` checks conditions in order and takes the first that is true, exactly like `case_when`.

In [47]:
matches["phase"] = np.where(matches["round"] <= 6, "Early (Rds 0-6)", "Rest of Season")
matches["phase"].value_counts()

phase
Rest of Season     153
Early (Rds 0-6)     54
Name: count, dtype: int64

### Your turn 3.1: translate your own `case_when`

This is the `winning_team` logic from your `fetch_afl_data.R`:

```r
winning_team = case_when(
  home_score > away_score ~ home_team,
  away_score > home_score ~ away_team,
  TRUE ~ "Draw"
)
```

Add a `winning_team` column to `matches` using `np.select`. Then set `n_draws` to the number of draws.

In [48]:
matches["winning_team"] = np.select([matches["home_score"] > matches["away_score"], matches["home_score"] < matches["away_score"]], 
                                    [matches["home_team"], matches["away_team"]], 
                                    default="Draw")
n_draws = matches["winning_team"].value_counts().get("Draw", 0)     

In [49]:
check("3.1 n_draws", lambda: n_draws, 1)
check("3.1 first match winner", lambda: matches["winning_team"].iloc[0], 'Carlton')

PASS     3.1 n_draws
PASS     3.1 first match winner


### Your turn 3.2: translate your nickname lookup

Your R script used a named vector and `ifelse(is.na(nick), "", nick)`. In pandas, `.map(dict)` looks each value up and gives `NaN` when it is missing, and `.fillna("")` replaces those.

Add `home_nickname` using the dictionary below, which is copied from your R script. Then set `n_blank` to the number of rows where the nickname came out blank, and look at which team it is. That is a real naming mismatch: AFL Tables says "Brisbane Lions", and this dataset says "Brisbane".

In [50]:
nicknames = {
    'Adelaide': 'Crows',
    'Brisbane Lions': 'Lions',
    'Carlton': 'Blues',
    'Collingwood': 'Magpies',
    'Essendon': 'Bombers',
    'Fremantle': 'Dockers',
    'Geelong': 'Cats',
    'Gold Coast': 'Suns',
    'GWS': 'Giants',
    'Hawthorn': 'Hawks',
    'Melbourne': 'Demons',
    'North Melbourne': 'Kangaroos',
    'Port Adelaide': 'Power',
    'Richmond': 'Tigers',
    'St Kilda': 'Saints',
    'Sydney': 'Swans',
    'West Coast': 'Eagles',
    'Western Bulldogs': 'Bulldogs',
}

In [51]:
matches["home_nickname"] = matches["home_team"].map(nicknames)
n_blank = matches["home_nickname"].isnull().sum()

In [52]:
check("3.2 n_blank", lambda: n_blank, 12)
print(matches.loc[matches["home_nickname"] == "", "home_team"].unique())

PASS     3.2 n_blank
<StringArray>
[]
Length: 0, dtype: str


### Your turn 3.3

Create `renamed`: a copy of `matches` with `home_team` renamed to `home` and `away_team` renamed to `away`. Do not change `matches` itself.

In [53]:
renamed = matches.rename(columns={"home_team": "home", "away_team": "away"})

In [54]:
check("3.3 new names", lambda: {"home", "away"} <= set(renamed.columns), True)
check("3.3 old names gone", lambda: "home_team" in renamed.columns, False)
check("3.3 matches untouched", lambda: "home_team" in matches.columns, True)

PASS     3.3 new names
PASS     3.3 old names gone
PASS     3.3 matches untouched


## 4. Translate five of your own snippets

Choose five short dplyr snippets from your own R code: your app and fetch script are full of good candidates (`r1_game_counts`, the `round_name` recode, `phase_summary`). Use the cells below to write each in pandas against `matches`, with the R version as a comment above it.

There are no checks here. Compare the output with what R gives you.

In [55]:
# Snippet 1 (R):
#completed <- match_data %>%
#      group_by(season) %>%
#      summarise(
#        has_early = any(round <= 6),
#        has_late  = any(round > 6),
#        .groups   = "drop"
#      ) %>%
#      filter(has_early, has_late) %>%
#      slice_max(season, n = 1)
# pandas:
completed = (
    matches
    .groupby("season", as_index=False)
    .agg(has_early=("round", lambda x: (x <= 6).any()),
         has_late=("round", lambda x: (x > 6).any()))
    .query("has_early and has_late")
    .nlargest(1, "season")
)

completed

,season,has_early,has_late
0,2025,True,True


In [56]:
# Snippet 2 (R):
# early_avgs_df <- match_data %>%
#      filter(round <= 6) %>%
#      group_by(season) %>%
#      summarise(early_avg = mean(avg_team_score), .groups = "drop")
# pandas:
early_avgs_df = (
    matches
    .query("round <= 6")
    .groupby("season", as_index=False)
    .agg(early_avg=("away_score", "mean"))
)
early_avgs_df

,season,early_avg
0,2025,77.888889


## Wrap-up

Run All, confirm every check passes, then commit from VS Code's Source Control panel (see the **Git in VS Code** tab) with the message `Session 7: dplyr verbs in pandas`.